# Debug benchmark consult resolution

Notebook to debug and fix consult resolution in the Prolog benchmark analyzer.

1. Import required libraries and benchmark analyzer
2. Reproduce undefined predicate diagnostics for `4.1.pl`
3. Inspect consult path resolution and symbol table consults
4. Fix consulted file loading and relative path resolution
5. Verify the fix with benchmark rerun

In [13]:
from pls.benchmark_runner import BenchmarkAnalyzer
from pls.prolog_visitor import PrologVisitor, Opts
from pls.utils import file_uri_to_path, path_to_file_uri
from pathlib import Path

print('imports ok')

imports ok


In [14]:
analyzer = BenchmarkAnalyzer()
result = analyzer.analyze_file('student_answers_MT-recurso/1/4.1.pl')
print('total_errors:', result['total_errors'])
print('errors_by_pass:', result['errors_by_pass'])
for detail in result['error_details']:
    print(detail)

total_errors: 0
errors_by_pass: {}


In [15]:
file_path = Path('student_answers_MT-recurso/1/4.1.pl')
uri = path_to_file_uri(file_path)
source = file_path.read_text(encoding='utf-8')

from tree_sitter import Parser, Language
from tree_sitter_prolog import prolog

PROLOG = Language(prolog())
parser = Parser(PROLOG)
tree = parser.parse(bytes(source, 'utf-8'))
visitor = PrologVisitor(uri)
visitor.visit(tree.root_node, Opts())

print('consult_paths:', visitor.consult_paths)
for consult_uri in visitor.consult_paths:
    print('consult_uri', consult_uri)
    print('consult_file exists?', file_uri_to_path(consult_uri).exists())

consult_paths: defaultdict(<class 'list'>, {'file:///home/pfpo/pls/student_answers_MT-recurso/1/common.pl': [file:///home/pfpo/pls/student_answers_MT-recurso/1/4.1.pl:3:3-3:23]})
consult_uri file:///home/pfpo/pls/student_answers_MT-recurso/1/common.pl
consult_file exists? True


/tmp/ipykernel_15362/2362044889.py:8: DeprecationWarning: int argument support is deprecated
  PROLOG = Language(prolog())


In [16]:
# inspect the consults loaded by BenchmarkAnalyzer
from pathlib import Path
from pls.benchmark_runner import BenchmarkAnalyzer
from pls.utils import path_to_file_uri

analyzer = BenchmarkAnalyzer()
file_path = Path('student_answers_MT-recurso/1/4.1.pl')
uri = path_to_file_uri(file_path)

# Reconstruct the same loading path to inspect internal tables
source = file_path.read_text(encoding='utf-8', errors='replace')
from tree_sitter import Parser, Language
from tree_sitter_prolog import prolog

PROLOG = Language(prolog())
parser = Parser(PROLOG)
tree = parser.parse(bytes(source, 'utf-8'))
from pls.prolog_visitor import PrologVisitor, Opts
visitor = PrologVisitor(uri)
visitor.visit(tree.root_node, Opts())
root_table = visitor

# print consult table details from benchmark runner
result_table = analyzer._load_builtins()
print('common.pl consult by path exists? should be true')
print('root consult_paths:', visitor.consult_paths)


common.pl consult by path exists? should be true
root consult_paths: defaultdict(<class 'list'>, {'file:///home/pfpo/pls/student_answers_MT-recurso/1/common.pl': [file:///home/pfpo/pls/student_answers_MT-recurso/1/4.1.pl:3:3-3:23]})


/tmp/ipykernel_15362/318019702.py:15: DeprecationWarning: int argument support is deprecated
  PROLOG = Language(prolog())


## Fix consulted file loading

We need to ensure `add_paths` resolves relative `consult('common.pl')` against the parent directory of the source file rather than the file path itself.

In [17]:
from pls.utils import add_paths
file_uri = path_to_file_uri(Path('student_answers_MT-recurso/1/4.1.pl'))
print('resolved common.pl:', add_paths(file_uri, 'common.pl'))
print('exists?', file_uri_to_path(add_paths(file_uri, 'common.pl')).exists())

resolved common.pl: file:///home/pfpo/pls/student_answers_MT-recurso/1/common.pl
exists? True


## Verify the fix with benchmark rerun

Rerun the benchmark on `4.1.pl` and verify that consulted predicates from `common.pl` are now recognized.

In [18]:
analyzer = BenchmarkAnalyzer()
result = analyzer.analyze_file('student_answers_MT-recurso/1/4.1.pl')
print('total_errors after fix:', result['total_errors'])
print('errors_by_pass after fix:', result['errors_by_pass'])
for detail in result['error_details']:
    print(detail)

total_errors after fix: 0
errors_by_pass after fix: {}


In [19]:
import importlib
import pls.utils
import pls.benchmark_runner

importlib.reload(pls.utils)
importlib.reload(pls.benchmark_runner)

from pls.benchmark_runner import BenchmarkAnalyzer

analyzer = BenchmarkAnalyzer()
result = analyzer.analyze_file('student_answers_MT-recurso/1/4.1.pl')
print('total_errors after code reload:', result['total_errors'])
print('errors_by_pass after code reload:', result['errors_by_pass'])
for detail in result['error_details']:
    print(detail)


total_errors after code reload: 0
errors_by_pass after code reload: {}
